In [2]:
import pandas as pd
import numpy as np
import joblib
import json
import warnings


# Load trained models
xgb_model = joblib.load("models/xgb_model.pkl")
lgb_model = joblib.load("models/lgb_model.pkl")

# Load model configuration
with open("models/model_config.json", "r") as f:
    model_config = json.load(f)

THRESHOLD    = model_config["threshold"]
XGB_WEIGHT   = model_config["xgb_weight"]
LGB_WEIGHT   = model_config["lgb_weight"]
FEATURE_COLS = model_config["feature_cols"]

print("Models and config loaded")
print()
print(f"  Threshold:    {THRESHOLD}")
print(f"  XGB weight:   {XGB_WEIGHT}")
print(f"  LGB weight:   {LGB_WEIGHT}")
print(f"  Features:     {len(FEATURE_COLS)}")

Models and config loaded

  Threshold:    0.4
  XGB weight:   0.55
  LGB weight:   0.45
  Features:     54


In [3]:
def validate_rider_data(rider_features):
    """
    Station 1: Validate incoming rider data.
    Returns (is_valid, reason) tuple.

    rider_features: dict of feature name → value
    """
    errors   = []
    warnings = []

    # ── Check 1: Minimum income evidence ────────────────────────────────────
    if rider_features.get("avg_daily_income", 0) <= 0:
        errors.append("No income transactions found — minimum 90 days M-Pesa history required")

    # ── Check 2: Income not unrealistically high ─────────────────────────────
    if rider_features.get("avg_daily_income", 0) > 10000:
        warnings.append("Unusually high daily income detected — manual verification recommended")

    # ── Check 3: NDI is positive ─────────────────────────────────────────────
    if rider_features.get("estimated_ndi", 0) <= 0:
        errors.append("Estimated net disposable income is zero or negative — cannot recommend loan")

    # ── Check 4: Debt service ratio not catastrophic ─────────────────────────
    if rider_features.get("debt_service_ratio", 0) > 0.80:
        errors.append("Existing debt obligations exceed 80% of income — over-indebtedness detected")

    # ── Check 5: Repayment to NDI ratio ─────────────────────────────────────
    if rider_features.get("repayment_to_ndi_ratio", 0) > 5.0:
        warnings.append("Requested amount is very high relative to disposable income")

    # ── Result ────────────────────────────────────────────────────────────────
    is_valid = len(errors) == 0

    return {
        "is_valid": is_valid,
        "errors":   errors,
        "warnings": warnings,
    }

print("✅ Station 1 — Data Validation defined")

✅ Station 1 — Data Validation defined


In [4]:
def prepare_features(rider_features):
    """
    Station 2: Prepare feature vector for model input.
    Ensures all 54 features are present in correct order.
    Missing features get safe default values.

    rider_features: dict of feature name → value
    Returns: numpy array of shape (1, 54)
    """
    feature_vector = []

    # Safe defaults for each feature type
    # These represent a neutral/average rider
    safe_defaults = {
        # Income features
        "avg_daily_income":         800.0,
        "total_income_90d":         72000.0,
        "income_volatility_cv":     0.35,
        "income_trend_90d":         0.0,
        "rain_season_dip":          0.20,
        "income_gap_max_days":      3,

        # Expense features
        "monthly_income_estimate":  20800.0,
        "fuel_spend_monthly":       7000.0,
        "fuel_to_income_ratio":     0.30,
        "sacco_contrib_monthly":    0.0,
        "family_remittance_monthly":2000.0,
        "digital_loan_monthly_out": 0.0,
        "estimated_ndi":            8000.0,
        "debt_service_ratio":       0.15,
        "max_safe_repayment":       2400.0,
        "avg_mpesa_balance":        1500.0,
        "min_mpesa_balance":        0.0,
        "zero_balance_days":        5,

        # Savings features
        "sacco_tenure_months":      0,
        "sacco_contribution_rate":  0.0,
        "sacco_on_time_rate":       0.0,
        "sacco_avg_days_late":      0.0,
        "sacco_cumulative_savings": 0.0,
        "sacco_guarantor_count":    0,
        "sacco_repayment_status":   2.0,
        "is_sacco_member":          0,

        # Loan history features
        "total_loans_taken":        0,
        "loans_repaid_clean":       0,
        "on_time_repayment_rate":   0.75,
        "avg_days_early_late":      0.0,
        "max_loan_repaid":          0.0,
        "active_digital_loans":     0,
        "digital_loan_frequency":   0,
        "ever_defaulted":           0,
        "restructured_loan_flag":   0,
        "total_outstanding_debt":   0.0,
        "first_time_borrower":      1,

        # Behavioural features
        "age":                      28,
        "months_operating":         12,
        "bike_age_years":           3.0,
        "bike_owned":               0,
        "has_app":                  0,
        "segment_risk_score":       3,
        "app_avg_weekly_trips":     35.0,
        "app_avg_weekly_earnings":  5000.0,
        "app_platform_rating":      4.5,
        "app_cancellation_rate":    0.06,
        "app_active_days_avg":      5.5,
        "app_peak_hour_ratio":      0.44,
        "app_income_stability_cv":  0.30,

        # Loan request features
        "loan_purpose_code":        3,
        "loan_to_income_ratio":     1.0,
        "loan_to_ndi_ratio":        2.0,
        "repayment_to_ndi_ratio":   0.60,
    }

    # Build feature vector in exact order model expects
    for feature in FEATURE_COLS:
        value = rider_features.get(feature, safe_defaults.get(feature, 0))
        feature_vector.append(float(value))

    return np.array(feature_vector).reshape(1, -1)

print("✅ Station 2 — Feature Preparation defined")

✅ Station 2 — Feature Preparation defined


In [5]:
def run_ml_scoring(feature_array):
    """
    Station 3: Run XGBoost and LightGBM, combine predictions.
    Station 4: Convert probability to credit score 0-100.

    Returns dict with probability, credit score and risk tier.
    """

    # ── Station 3: ML predictions ────────────────────────────────────────────
    xgb_proba = xgb_model.predict_proba(feature_array)[0][1]
    lgb_proba = lgb_model.predict_proba(feature_array)[0][1]

    # Ensemble weighted average
    pd_score = (xgb_proba * XGB_WEIGHT) + (lgb_proba * LGB_WEIGHT)
    pd_score = float(np.clip(pd_score, 0.01, 0.99))

    # ── Station 4: Convert to credit score ───────────────────────────────────
    # Higher credit score = lower default probability
    # Scale: 0 = certain default, 100 = certain repayment
    credit_score = round(100 - (pd_score * 100), 1)

    # ── Risk tier classification ──────────────────────────────────────────────
    if credit_score >= 80:
        risk_tier   = "LOW"
        risk_label  = " Low Risk"
        action      = "APPROVE"
    elif credit_score >= 65:
        risk_tier   = "LOW-MEDIUM"
        risk_label  = "Low-Medium Risk"
        action      = "APPROVE WITH CONDITIONS"
    elif credit_score >= 50:
        risk_tier   = "MEDIUM"
        risk_label  = " Medium Risk"
        action      = "CONDITIONAL APPROVAL"
    elif credit_score >= 35:
        risk_tier   = "HIGH"
        risk_label  = " High Risk"
        action      = "SENIOR REVIEW REQUIRED"
    else:
        risk_tier   = "VERY HIGH"
        risk_label  = " Very High Risk"
        action      = "DECLINE"

    return {
        "xgb_probability":  round(xgb_proba, 4),
        "lgb_probability":  round(lgb_proba, 4),
        "pd_score":         round(pd_score, 4),
        "pd_percentage":    f"{pd_score*100:.1f}%",
        "credit_score":     credit_score,
        "risk_tier":        risk_tier,
        "risk_label":       risk_label,
        "action":           action,
    }

print("✅ Station 3 and 4 — ML Scoring and Score Conversion defined")

✅ Station 3 and 4 — ML Scoring and Score Conversion defined


In [6]:
def calculate_loan_recommendation(rider_features, scoring_result,
                                   requested_amount, requested_term_days):
    """
    Station 5: Calculate loan amount and term recommendation.

    Uses NDI, credit score, and loan history to determine
    the safest loan amount and term for this specific rider.
    """

    credit_score  = scoring_result["credit_score"]
    risk_tier     = scoring_result["risk_tier"]
    estimated_ndi = rider_features.get("estimated_ndi", 8000)
    first_timer   = rider_features.get("first_time_borrower", 1)
    max_repaid    = rider_features.get("max_loan_repaid", 0)

    # ── Hard block — decline immediately ─────────────────────────────────────
    if risk_tier == "VERY HIGH":
        return {
            "recommended_amount":    0,
            "recommended_term_days": 0,
            "monthly_repayment":     0,
            "daily_repayment":       0,
            "decision":              "DECLINE",
            "decline_reason":        "Credit score below minimum threshold",
            "condition":             None,
        }

    # ── Step 1: Maximum safe monthly repayment ────────────────────────────────
    # 30% of NDI is the absolute ceiling
    max_monthly_repayment = estimated_ndi * 0.30

    # ── Step 2: Credit score adjustment factor ────────────────────────────────
    score_factors = {
        "LOW":         1.00,
        "LOW-MEDIUM":  0.85,
        "MEDIUM":      0.65,
        "HIGH":        0.45,
    }
    score_factor = score_factors.get(risk_tier, 0.65)

    # ── Step 3: Adjusted monthly repayment ───────────────────────────────────
    adjusted_repayment = max_monthly_repayment * score_factor

    # ── Step 4: Determine recommended term ───────────────────────────────────
    # Base term from request
    term_months = requested_term_days / 30

    # First time borrowers get shorter terms
    if first_timer:
        term_months = min(term_months, 3)

    # High risk riders get shorter terms
    if risk_tier == "HIGH":
        term_months = min(term_months, 2)

    term_months = max(term_months, 1)    # minimum 1 month
    term_days   = int(term_months * 30)

    # ── Step 5: Calculate recommended loan amount ─────────────────────────────
    recommended_amount = adjusted_repayment * term_months

    # ── Step 6: Apply hard limits ─────────────────────────────────────────────
    # First time borrowers — maximum KES 15,000
    if first_timer:
        recommended_amount = min(recommended_amount, 15000)

    # Cannot exceed proven repayment capacity
    # If they repaid KES 10,000 before, do not give KES 50,000
    if max_repaid > 0:
        recommended_amount = min(recommended_amount, max_repaid * 1.5)

    # Minimum loan amount
    recommended_amount = max(recommended_amount, 3000)

    # Round to nearest KES 500
    recommended_amount = round(recommended_amount / 500) * 500

    # Do not exceed what they requested
    recommended_amount = min(recommended_amount, requested_amount)

    # ── Step 7: Calculate repayment amounts ──────────────────────────────────
    monthly_repayment = recommended_amount / term_months
    daily_repayment   = monthly_repayment / 30

    # ── Step 8: Determine conditions ─────────────────────────────────────────
    conditions = []

    if first_timer:
        conditions.append("Guarantor required — first loan")
    if risk_tier in ["MEDIUM", "HIGH"]:
        conditions.append("Guarantor required")
    if risk_tier == "HIGH":
        conditions.append("Senior officer approval required")
    if recommended_amount < requested_amount:
        conditions.append(
            f"Amount reduced from KES {requested_amount:,} "
            f"to KES {recommended_amount:,} based on repayment capacity"
        )

    condition_text = " | ".join(conditions) if conditions else "None"

    # ── Decision ──────────────────────────────────────────────────────────────
    if risk_tier == "HIGH":
        decision = "SENIOR REVIEW REQUIRED"
    elif conditions:
        decision = "CONDITIONALLY APPROVED"
    else:
        decision = "APPROVED"

    return {
        "recommended_amount":    int(recommended_amount),
        "recommended_term_days": term_days,
        "monthly_repayment":     round(monthly_repayment, 2),
        "daily_repayment":       round(daily_repayment, 2),
        "decision":              decision,
        "decline_reason":        None,
        "condition":             condition_text,
    }

print("✅ Station 5 — Loan Recommendation Engine defined")

✅ Station 5 — Loan Recommendation Engine defined


In [7]:
def score_rider(rider_features, requested_amount, requested_term_days,
                rider_name="Rider"):
    """
    Station 6: Complete scoring pipeline.
    Runs all 5 stations and returns the full credit decision.

    rider_features:      dict of feature name → value
    requested_amount:    KES amount the rider wants to borrow
    requested_term_days: how many days they want to repay over
    rider_name:          name for the output report
    """

    # ── Add loan request features ─────────────────────────────────────────────
    monthly_income   = rider_features.get("monthly_income_estimate", 20800)
    estimated_ndi    = rider_features.get("estimated_ndi", 8000)
    requested_monthly = requested_amount / max(requested_term_days / 30, 1)

    rider_features["requested_amount"]      = requested_amount
    rider_features["requested_term_days"]   = requested_term_days
    rider_features["loan_to_income_ratio"]  = round(
        requested_amount / max(monthly_income, 1), 4)
    rider_features["loan_to_ndi_ratio"]     = round(
        requested_amount / max(estimated_ndi, 1), 4)
    rider_features["repayment_to_ndi_ratio"]= round(
        requested_monthly / max(estimated_ndi, 1), 4)
    rider_features["loan_purpose_code"]     = rider_features.get(
        "loan_purpose_code", 3)

    # ── Station 1: Validate ───────────────────────────────────────────────────
    validation = validate_rider_data(rider_features)

    if not validation["is_valid"]:
        return {
            "rider_name":    rider_name,
            "valid":         False,
            "errors":        validation["errors"],
            "decision":      "REJECTED — INSUFFICIENT DATA",
            "credit_score":  None,
            "pd_score":      None,
            "recommendation":None,
        }

    # ── Station 2: Prepare features ───────────────────────────────────────────
    feature_array = prepare_features(rider_features)

    # ── Station 3 and 4: Score ────────────────────────────────────────────────
    scoring = run_ml_scoring(feature_array)

    # ── Station 5: Loan recommendation ───────────────────────────────────────
    recommendation = calculate_loan_recommendation(
        rider_features, scoring,
        requested_amount, requested_term_days
    )

    return {
        "rider_name":    rider_name,
        "valid":         True,
        "errors":        [],
        "warnings":      validation["warnings"],
        "scoring":       scoring,
        "recommendation":recommendation,
    }

print("✅ Station 6 — Complete Score Function defined")

✅ Station 6 — Complete Score Function defined


In [8]:
def print_credit_decision(result, rider_name, requested_amount,
                           requested_term_days):
    """
    Format and print the complete credit decision
    exactly as a loan officer would see it.
    """

    print()
    print("═" * 55)
    print("  BODACREDIT SCORE — LOAN DECISION")
    print("═" * 55)
    print(f"  Rider:            {rider_name}")
    print(f"  Requested:        KES {requested_amount:,} / "
          f"{requested_term_days} days")
    print("─" * 55)

    if not result["valid"]:
        print(f"  ❌ APPLICATION REJECTED")
        print()
        for error in result["errors"]:
            print(f"  Reason: {error}")
        print("═" * 55)
        return

    s = result["scoring"]
    r = result["recommendation"]

    # ── Credit Score ──────────────────────────────────────────────────────────
    print(f"  CREDIT SCORE:     {s['credit_score']} / 100")
    print(f"  RISK:             {s['risk_label']}")
    print(f"  DEFAULT PROB:     {s['pd_percentage']}")
    print("─" * 55)

    # ── Decision ──────────────────────────────────────────────────────────────
    if r["decision"] == "DECLINE":
        print(f"  DECISION:         ❌ DECLINED")
        print(f"  Reason:           {r['decline_reason']}")
    else:
        decision_icon = "✅" if r["decision"] == "APPROVED" else "⚠️ "
        print(f"  DECISION:         {decision_icon} {r['decision']}")
        print()
        print(f"  Approved Amount:  KES {r['recommended_amount']:,}")
        print(f"  Term:             {r['recommended_term_days']} days")
        print(f"  Monthly Payment:  KES {r['monthly_repayment']:,.0f}")
        print(f"  Daily M-Pesa:     KES {r['daily_repayment']:,.0f} / day")
        if r["condition"] and r["condition"] != "None":
            print()
            print(f"  Conditions:")
            for cond in r["condition"].split(" | "):
                print(f"    • {cond}")

    print("═" * 55)

print("✅ Output formatter defined")

✅ Output formatter defined


In [9]:
print("TESTING SCORING PIPELINE ON 3 SAMPLE RIDERS")
print()

# ── Rider 1: Strong SACCO member, good history ───────────────────────────────
rider_1 = {
    "avg_daily_income":         950.0,
    "total_income_90d":         85500.0,
    "income_volatility_cv":     0.18,
    "income_trend_90d":         0.04,
    "rain_season_dip":          0.22,
    "income_gap_max_days":      2,
    "monthly_income_estimate":  24700.0,
    "fuel_spend_monthly":       7800.0,
    "fuel_to_income_ratio":     0.32,
    "sacco_contrib_monthly":    2500.0,
    "family_remittance_monthly":2000.0,
    "digital_loan_monthly_out": 0.0,
    "estimated_ndi":            12400.0,
    "debt_service_ratio":       0.10,
    "max_safe_repayment":       3720.0,
    "avg_mpesa_balance":        1840.0,
    "min_mpesa_balance":        120.0,
    "zero_balance_days":        2,
    "sacco_tenure_months":      14,
    "sacco_contribution_rate":  0.93,
    "sacco_on_time_rate":       0.86,
    "sacco_avg_days_late":      0.8,
    "sacco_cumulative_savings": 35000.0,
    "sacco_guarantor_count":    2,
    "sacco_repayment_status":   2.0,
    "is_sacco_member":          1,
    "total_loans_taken":        3,
    "loans_repaid_clean":       3,
    "on_time_repayment_rate":   1.0,
    "avg_days_early_late":      -1.5,
    "max_loan_repaid":          15000.0,
    "active_digital_loans":     0,
    "digital_loan_frequency":   0,
    "ever_defaulted":           0,
    "restructured_loan_flag":   0,
    "total_outstanding_debt":   0.0,
    "first_time_borrower":      0,
    "age":                      29,
    "months_operating":         24,
    "bike_age_years":           2.5,
    "bike_owned":               1,
    "has_app":                  1,
    "segment_risk_score":       1,
    "app_avg_weekly_trips":     42.0,
    "app_avg_weekly_earnings":  6200.0,
    "app_platform_rating":      4.7,
    "app_cancellation_rate":    0.04,
    "app_active_days_avg":      5.8,
    "app_peak_hour_ratio":      0.44,
    "app_income_stability_cv":  0.15,
}

result_1 = score_rider(rider_1, requested_amount=20000,
                        requested_term_days=90,
                        rider_name="John Kamau — SACCO Member")
print_credit_decision(result_1, "John Kamau — SACCO Member", 20000, 90)

print()

# ── Rider 2: Hired stage rider, no SACCO, digital loans ──────────────────────
rider_2 = {
    "avg_daily_income":         620.0,
    "total_income_90d":         55800.0,
    "income_volatility_cv":     0.52,
    "income_trend_90d":        -0.03,
    "rain_season_dip":          0.35,
    "income_gap_max_days":      8,
    "monthly_income_estimate":  16120.0,
    "fuel_spend_monthly":       8500.0,
    "fuel_to_income_ratio":     0.53,
    "sacco_contrib_monthly":    0.0,
    "family_remittance_monthly":1500.0,
    "digital_loan_monthly_out": 1200.0,
    "estimated_ndi":            3000.0,
    "debt_service_ratio":       0.45,
    "max_safe_repayment":       900.0,
    "avg_mpesa_balance":        320.0,
    "min_mpesa_balance":        0.0,
    "zero_balance_days":        18,
    "sacco_tenure_months":      0,
    "sacco_contribution_rate":  0.0,
    "sacco_on_time_rate":       0.0,
    "sacco_avg_days_late":      0.0,
    "sacco_cumulative_savings": 0.0,
    "sacco_guarantor_count":    0,
    "sacco_repayment_status":   2.0,
    "is_sacco_member":          0,
    "total_loans_taken":        4,
    "loans_repaid_clean":       1,
    "on_time_repayment_rate":   0.25,
    "avg_days_early_late":      18.0,
    "max_loan_repaid":          3000.0,
    "active_digital_loans":     2,
    "digital_loan_frequency":   4,
    "ever_defaulted":           1,
    "restructured_loan_flag":   1,
    "total_outstanding_debt":   8500.0,
    "first_time_borrower":      0,
    "age":                      24,
    "months_operating":         6,
    "bike_age_years":           6.5,
    "bike_owned":               0,
    "has_app":                  0,
    "segment_risk_score":       5,
    "app_avg_weekly_trips":     35.0,
    "app_avg_weekly_earnings":  5000.0,
    "app_platform_rating":      4.5,
    "app_cancellation_rate":    0.06,
    "app_active_days_avg":      5.5,
    "app_peak_hour_ratio":      0.44,
    "app_income_stability_cv":  0.30,
}

result_2 = score_rider(rider_2, requested_amount=15000,
                        requested_term_days=60,
                        rider_name="Peter Otieno — Stage Rider")
print_credit_decision(result_2, "Peter Otieno — Stage Rider", 15000, 60)

print()

# ── Rider 3: First time borrower — app rider ──────────────────────────────────
rider_3 = {
    "avg_daily_income":         820.0,
    "total_income_90d":         73800.0,
    "income_volatility_cv":     0.25,
    "income_trend_90d":         0.02,
    "rain_season_dip":          0.18,
    "income_gap_max_days":      3,
    "monthly_income_estimate":  21320.0,
    "fuel_spend_monthly":       7200.0,
    "fuel_to_income_ratio":     0.34,
    "sacco_contrib_monthly":    0.0,
    "family_remittance_monthly":1800.0,
    "digital_loan_monthly_out": 0.0,
    "estimated_ndi":            9500.0,
    "debt_service_ratio":       0.08,
    "max_safe_repayment":       2850.0,
    "avg_mpesa_balance":        1200.0,
    "min_mpesa_balance":        50.0,
    "zero_balance_days":        4,
    "sacco_tenure_months":      0,
    "sacco_contribution_rate":  0.0,
    "sacco_on_time_rate":       0.0,
    "sacco_avg_days_late":      0.0,
    "sacco_cumulative_savings": 0.0,
    "sacco_guarantor_count":    0,
    "sacco_repayment_status":   2.0,
    "is_sacco_member":          0,
    "total_loans_taken":        0,
    "loans_repaid_clean":       0,
    "on_time_repayment_rate":   0.75,
    "avg_days_early_late":      0.0,
    "max_loan_repaid":          0.0,
    "active_digital_loans":     0,
    "digital_loan_frequency":   0,
    "ever_defaulted":           0,
    "restructured_loan_flag":   0,
    "total_outstanding_debt":   0.0,
    "first_time_borrower":      1,
    "age":                      26,
    "months_operating":         8,
    "bike_age_years":           1.5,
    "bike_owned":               1,
    "has_app":                  1,
    "segment_risk_score":       3,
    "app_avg_weekly_trips":     38.0,
    "app_avg_weekly_earnings":  5800.0,
    "app_platform_rating":      4.6,
    "app_cancellation_rate":    0.05,
    "app_active_days_avg":      5.6,
    "app_peak_hour_ratio":      0.46,
    "app_income_stability_cv":  0.20,
}

result_3 = score_rider(rider_3, requested_amount=10000,
                        requested_term_days=60,
                        rider_name="James Mwangi — First Time Borrower")
print_credit_decision(result_3, "James Mwangi — First Time Borrower",
                      10000, 60)

TESTING SCORING PIPELINE ON 3 SAMPLE RIDERS


═══════════════════════════════════════════════════════
  BODACREDIT SCORE — LOAN DECISION
═══════════════════════════════════════════════════════
  Rider:            John Kamau — SACCO Member
  Requested:        KES 20,000 / 90 days
───────────────────────────────────────────────────────
  CREDIT SCORE:     69.5 / 100
  RISK:             Low-Medium Risk
  DEFAULT PROB:     30.5%
───────────────────────────────────────────────────────
  DECISION:         ⚠️  CONDITIONALLY APPROVED

  Approved Amount:  KES 9,500
  Term:             90 days
  Monthly Payment:  KES 3,167
  Daily M-Pesa:     KES 106 / day

  Conditions:
    • Amount reduced from KES 20,000 to KES 9,500 based on repayment capacity
═══════════════════════════════════════════════════════


═══════════════════════════════════════════════════════
  BODACREDIT SCORE — LOAN DECISION
═══════════════════════════════════════════════════════
  Rider:            Peter Otieno — Stage Rider


In [10]:
import pickle

# Save the complete pipeline as a single object
pipeline = {
    "xgb_model":      xgb_model,
    "lgb_model":      lgb_model,
    "model_config":   model_config,
    "functions": {
        "validate_rider_data":          validate_rider_data,
        "prepare_features":             prepare_features,
        "run_ml_scoring":               run_ml_scoring,
        "calculate_loan_recommendation":calculate_loan_recommendation,
        "score_rider":                  score_rider,
        "print_credit_decision":        print_credit_decision,
    }
}

with open("models/scoring_pipeline.pkl", "wb") as f:
    pickle.dump(pipeline, f)

print("✅ Complete scoring pipeline saved")
print("   models/scoring_pipeline.pkl")
print()
print("=" * 45)
print("  CODE PHASE 4 COMPLETE")
print("  Next: Code Phase 5 — SHAP Explainability")
print("=" * 45)

✅ Complete scoring pipeline saved
   models/scoring_pipeline.pkl

  CODE PHASE 4 COMPLETE
  Next: Code Phase 5 — SHAP Explainability
